# Ingest Vehicle Test Data → Unity Catalog (Optimized)

把 Azure Blob Storage 容器 `vehicle-test-data` 中的 3 个路径抽取到当前 DBR 的 **默认 UC managed 存储**（即 metastore root）下，注册为 Delta 表。

这版 notebook 提供两种读取模式：

- `direct`: 推荐给普通 compute cluster。Spark executor 并行直读 Blob，速度最快。
- `staged`: 兼容 serverless。Blob HTTPS + SAS 先下载到 UC managed volume，再写入 Delta。此模式已改成多线程下载，但仍然比 direct 慢。

| Delta 表名 | 源路径（容器内相对路径） |
|---|---|
| `tags_combine_final_source` | `ads_db_tag/tags_combine_final/Subject/*` |
| `driving_behavior_effective_mileage_source` | `edw_db_vehicle/dwd_tbl_driving_behavior_effective_mileage/*` |
| `tbl_vehicle_source` | `edw_db_vehicle/dwd_tbl_vehicle/pt=20260210/*` |

- **Catalog**: `vehicel_test_source`
- **Schema**: `Oringinal`
- **临时 managed volume**: `vehicle_ingest_stage`（只在 staged 模式使用）
- **要求**: 数据与 schema 保持原样、不做业务转换。
- **最终存储**: `saveAsTable(...)` 不指定 `LOCATION`，所以 Delta 表物理文件落在 metastore root 下。

In [ ]:
# ============================================================
# 1. 读取模式与 Azure Blob Storage SAS 配置
# ============================================================
# direct:  普通 compute cluster 推荐。Spark executor 并行直读 Blob，最快。
# staged:  serverless 兼容。先用 HTTPS + SAS 下载到 UC managed volume，再由 Spark 读取。
# auto:    先尝试 direct；如果当前环境禁止 fs.azure.sas.* 配置，则自动降级 staged。

import os
import shutil
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor, as_completed

STORAGE_ACCOUNT = "foundryhubtarhonestorage"
CONTAINER = "vehicle-test-data"

# compute cluster 建议用 direct；serverless 建议用 auto 或 staged。
INGEST_MODE = "auto"  # "direct" / "staged" / "auto"
DOWNLOAD_WORKERS = 32

# 注意：只放 "?" 后面的 query 部分，不要带前导问号。
SAS_TOKEN = "sp=racwdli&st=2026-05-14T04:33:17Z&se=2026-12-31T12:48:17Z&sv=2025-11-05&sr=c&sig=mRZCKJEj4zc%2BpYSmRvalgf8EzQpt60oEKiVgvE0ckKE%3D"

SOURCE_PREFIXES = {
    "tags_combine_final_source": "ads_db_tag/tags_combine_final/Subject/",
    "driving_behavior_effective_mileage_source": "edw_db_vehicle/dwd_tbl_driving_behavior_effective_mileage/",
    "tbl_vehicle_source": "edw_db_vehicle/dwd_tbl_vehicle/pt=20260210/",
}

BASE_HTTPS_URL = f"https://{STORAGE_ACCOUNT}.blob.core.windows.net/{CONTAINER}"
BASE_WASBS_URI = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"


def blob_url(blob_name: str) -> str:
    encoded_name = urllib.parse.quote(blob_name, safe="/")
    return f"{BASE_HTTPS_URL}/{encoded_name}?{SAS_TOKEN}"


def configure_direct_blob_access() -> bool:
    conf_key = f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net"
    try:
        spark.conf.set(conf_key, SAS_TOKEN)
        print("[OK] Direct Blob access configured by spark.conf.set")
        return True
    except Exception as e:
        print("[WARN] 当前 compute 不允许在 notebook 中设置 fs.azure.sas.*")
        print(f"       {type(e).__name__}: {e}")
        print("       如果使用普通 compute cluster，可改成 Single User，或把以下配置放到 Cluster Spark Config 后重启：")
        print(f"       {conf_key} {SAS_TOKEN}")
        return False


if INGEST_MODE not in {"direct", "staged", "auto"}:
    raise ValueError("INGEST_MODE must be one of: direct, staged, auto")

direct_available = False if INGEST_MODE == "staged" else configure_direct_blob_access()
ACTIVE_INGEST_MODE = "direct" if direct_available else "staged"

if INGEST_MODE == "direct" and not direct_available:
    raise RuntimeError("INGEST_MODE='direct' 但当前环境无法配置 Blob SAS，请切换 Single User cluster 或使用 staged/auto。")

print("Active ingest mode:", ACTIVE_INGEST_MODE)
print("Source container:", BASE_HTTPS_URL)

In [ ]:
# ============================================================
# 2. 目标 UC catalog / schema，并按需创建 managed volume 暂存区
# ============================================================
CATALOG = "vehicel_test_source"
SCHEMA = "Oringinal"
STAGE_VOLUME = "vehicle_ingest_stage"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

STAGE_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{STAGE_VOLUME}"

if ACTIVE_INGEST_MODE == "staged":
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{STAGE_VOLUME}`")
    print("Stage root:", STAGE_ROOT)
else:
    print("Direct mode: no staging volume is needed")

display(spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema"))

In [ ]:
# ============================================================
# 3. 快速探活：按当前模式检查源路径
# ============================================================
def list_blobs(prefix: str) -> list[str]:
    blobs = []
    marker = ""

    while True:
        query = {
            "restype": "container",
            "comp": "list",
            "prefix": prefix,
        }
        if marker:
            query["marker"] = marker

        list_url = f"{BASE_HTTPS_URL}?{urllib.parse.urlencode(query)}&{SAS_TOKEN}"

        with urllib.request.urlopen(list_url) as response:
            xml_bytes = response.read()

        root = ET.fromstring(xml_bytes)

        for blob_node in root.findall(".//Blob"):
            name = blob_node.findtext("Name")
            if name and not name.endswith("/"):
                blobs.append(name)

        marker = root.findtext("NextMarker") or ""
        if not marker:
            break

    return blobs


def source_path(prefix: str) -> str:
    return f"{BASE_WASBS_URI}/{prefix}*"


source_blobs = {}
for table_name, prefix in SOURCE_PREFIXES.items():
    if ACTIVE_INGEST_MODE == "direct":
        path = source_path(prefix)
        try:
            files = dbutils.fs.ls(f"{BASE_WASBS_URI}/{prefix}")
            print(f"[OK]  {table_name}: direct path is readable, {len(files)} entries under {prefix}")
            print(f"      Spark read path: {path}")
        except Exception as e:
            print(f"[WARN] dbutils.fs.ls failed for {table_name}; Spark read may still work with wildcard.")
            print(f"       {e}")
            print(f"       Spark read path: {path}")
    else:
        try:
            blobs = list_blobs(prefix)
            source_blobs[table_name] = blobs
            print(f"[OK]  {table_name}: {len(blobs)} blobs under {prefix}")
            for sample in blobs[:3]:
                print(f"      sample: {sample}")
        except Exception as e:
            print(f"[ERR] {table_name}: {prefix}\n      {e}")

In [ ]:
# ============================================================
# 4. 读取源数据并保持原始 schema 写入 Delta managed table
# ============================================================
SOURCE_FORMAT = "parquet"
MERGE_PARQUET_SCHEMA = False  # 文件 schema 完全一致时保持 False，速度明显更快；存在 schema 演进时再改 True。
CLEANUP_STAGE_AFTER_INGEST = True


def spark_reader(fmt: str = SOURCE_FORMAT):
    reader = spark.read.format(fmt)
    if fmt == "parquet" and MERGE_PARQUET_SCHEMA:
        reader = reader.option("mergeSchema", "true")
    return reader


def download_blob(blob_name: str, prefix: str, stage_dir: str) -> str:
    relative_path = blob_name[len(prefix):]
    if not relative_path:
        return ""

    local_path = os.path.join(stage_dir, relative_path)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    with urllib.request.urlopen(blob_url(blob_name)) as response, open(local_path, "wb") as output_file:
        shutil.copyfileobj(response, output_file, length=16 * 1024 * 1024)

    return local_path


def stage_blobs(table_name: str, prefix: str, blob_names: list[str]) -> str:
    stage_dir = f"{STAGE_ROOT}/{table_name}"
    dbutils.fs.rm(stage_dir, recurse=True)
    os.makedirs(stage_dir, exist_ok=True)

    total = len(blob_names)
    if total == 0:
        raise ValueError(f"No blobs found for {table_name} under prefix {prefix}")

    print(f"  downloading with {DOWNLOAD_WORKERS} threads")
    completed = 0
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = [pool.submit(download_blob, blob_name, prefix, stage_dir) for blob_name in blob_names]
        for future in as_completed(futures):
            future.result()
            completed += 1
            if completed == total or completed % 100 == 0:
                print(f"  staged {completed:,}/{total:,} blobs for {table_name}")

    return stage_dir


def load_source(table_name: str, prefix: str, blob_names: list[str] | None, fmt: str = SOURCE_FORMAT):
    if ACTIVE_INGEST_MODE == "direct":
        path = source_path(prefix)
        print(f"  Spark direct read path : {path}")
        return spark_reader(fmt).load(path), None

    stage_dir = stage_blobs(table_name, prefix, blob_names or [])
    print(f"  staged path            : {stage_dir}")
    return spark_reader(fmt).load(stage_dir), stage_dir


def ingest(table_name: str, prefix: str, blob_names: list[str] | None = None, fmt: str = SOURCE_FORMAT) -> None:
    """direct: Spark 并行直读 Blob；staged: HTTPS + SAS 并行下载后读取。"""
    print(f"\n=== Ingesting {table_name} ===")
    print(f"  active mode   : {ACTIVE_INGEST_MODE}")
    print(f"  source prefix : {prefix}")
    print(f"  source format : {fmt}")
    if blob_names is not None:
        print(f"  blob count    : {len(blob_names):,}")

    df, stage_dir = load_source(table_name, prefix, blob_names, fmt)
    fqn = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(fqn))

    row_count = spark.table(fqn).count()
    print(f"  rows          : {row_count:,}")
    print(f"  table         : {fqn}")

    if stage_dir and CLEANUP_STAGE_AFTER_INGEST:
        dbutils.fs.rm(stage_dir, recurse=True)
        print("  staged files  : cleaned up")

In [ ]:
# ============================================================
# 5. 执行三张表的抽取
# ============================================================
for table_name, prefix in SOURCE_PREFIXES.items():
    blob_names = None if ACTIVE_INGEST_MODE == "direct" else (source_blobs.get(table_name) or list_blobs(prefix))
    ingest(table_name, prefix, blob_names)

In [ ]:
# ============================================================
# 6. 验证：列出新建的表 & 抽样查看 schema/数据
# ============================================================
display(spark.sql(f"SHOW TABLES IN `{CATALOG}`.`{SCHEMA}`"))

for table_name in SOURCE_PREFIXES.keys():
    fqn = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"
    print(f"\n--- {fqn} ---")
    spark.table(fqn).printSchema()
    display(spark.table(fqn).limit(5))

In [ ]:
# ============================================================
# 7. 查看 Delta 表的物理存储位置
#    验证最终 Delta managed table 落到了 metastore root 下
# ============================================================
for table_name in SOURCE_PREFIXES.keys():
    fqn = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"
    display(spark.sql(f"DESCRIBE DETAIL {fqn}").select("name", "format", "location", "numFiles", "sizeInBytes"))